# 01 — Keypoint Extraction
Extraction of MediaPipe Holistic keypoints from ISL dataset videos.
Optimized for [Kaggle: ISL Words with Landmarks](https://www.kaggle.com/datasets/kaushikyh/indian-sign-language-words-with-landmarks).

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/isl_project'
os.makedirs(PROJECT_ROOT, exist_ok=True)

# Add src to path
import sys
sys.path.insert(0, f'{PROJECT_ROOT}/src')

# Install ONLY packages not preinstalled on Colab
%pip install -q mediapipe==0.10.21 opencv-python-headless==4.11.0.86 scipy==1.13.0 pyyaml>=6.0 tqdm>=4.65.0 scikit-learn>=1.3.0 seaborn>=0.12.0

In [ ]:
# Path to the Kaggle dataset after uploading to Drive
# Structure: isl_project/data/raw/ProcessedData_vivit/{word}/{video}.mp4
VIDEO_DIR = f'{PROJECT_ROOT}/data/raw/ProcessedData_vivit'

if not os.path.exists(VIDEO_DIR):
    print(f'WARNING: VIDEO_DIR not found at {VIDEO_DIR}')
    print('Please ensure you uploaded the Kaggle dataset to this path.')

In [ ]:
# Build label map
import os, json
SPLITS_DIR = f'{PROJECT_ROOT}/data/splits'
os.makedirs(SPLITS_DIR, exist_ok=True)

from utils import build_label_map
if os.path.exists(VIDEO_DIR):
    label_map = build_label_map(VIDEO_DIR)
    print(f'Classes: {len(label_map)}')
else:
    print('Label map cannot be built until VIDEO_DIR exists.')

In [ ]:
# Create train/val/test splits
from utils import create_splits
if os.path.exists(VIDEO_DIR):
    create_splits(VIDEO_DIR, label_map)

In [ ]:
# Download Holistic Landmarker model task file
MODELS_DIR = f'{PROJECT_ROOT}/models'
os.makedirs(MODELS_DIR, exist_ok=True)
MODEL_PATH = f'{MODELS_DIR}/holistic_landmarker.task'

if not os.path.exists(MODEL_PATH):
    import urllib.request
    url = 'https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/1/holistic_landmarker.task'
    print(f'Downloading model to {MODEL_PATH}...')
    urllib.request.urlretrieve(url, MODEL_PATH)
    print('Done.')
else:
    print(f'Model already exists at {MODEL_PATH}')

In [ ]:
# Extract keypoints
# Note: The Kaggle dataset has 32-frame videos. We upsample them to 64 frames (default TARGET_LEN).
from extract import preprocess_dataset
NPY_DIR = f'{PROJECT_ROOT}/data/keypoints'

if os.path.exists(VIDEO_DIR):
    preprocess_dataset(
        video_dir=VIDEO_DIR,
        save_dir=NPY_DIR,
        label_map=label_map,
        model_path=MODEL_PATH
    )

In [ ]:
# Verify extraction
from extract import verify_keypoints
verify_keypoints(NPY_DIR)

In [ ]:
# Compute normalization stats from training set
import numpy as np
from utils import compute_stats

train_files = open(f'{SPLITS_DIR}/train.txt').read().splitlines()
mean, std = compute_stats(NPY_DIR, train_files)
print(f'Mean shape: {mean.shape}, Std shape: {std.shape}')